# DOME Flow Matching 无 Resample 训练流程

这个 notebook 用原始 nuScenes occupancy 数据训练 DOME / world model，不生成也不读取 `data/resampled_occ`。

流程：

1. 检查项目根目录、数据、checkpoint 和配置
2. 确认训练配置没有使用 resample dataset
3. 启动只训练 DOME 的 flow matching 训练
4. 打开 TensorBoard 看指标
5. 评估 checkpoint
6. 可视化预测结果

注意：OccVAE 在这个流程里只加载 checkpoint 并冻结，不会训练。真正更新参数的是 DOME / world model。

## 0. 进入项目根目录

在服务器上打开这个 notebook 时，先确认当前目录是 `dome-cfm` 项目根目录。下面这个 cell 会自动检查关键文件。

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import runpy

ROOT = Path.cwd()

# 如果 notebook 不是从项目根目录启动，可以手动改这里：
# ROOT = Path('/mnt/data2/whz/dome-cfm')

os.chdir(ROOT)
print('当前目录:', ROOT)
print('训练入口存在:', (ROOT / 'tools/train_diffusion.py').exists())
print('无 resample 配置存在:', (ROOT / 'config/train_dome.py').exists())

当前目录: /mnt/data2/whz/dome-cfm
训练入口存在: True
无 resample 配置存在: True


## 1. 配置本次实验路径

默认使用 `config/train_dome.py`，这个配置的训练集是原始 `data/nuscenes/`，不是 `data/resampled_occ`。

如果你的数据或 checkpoint 放在别的地方，只改下面这个 cell。

In [2]:
# 基础配置
CFG = './config/train_dome.py'
WORK_DIR = './work_dir/dome_flow_no_resample'
VAE_CKPT = 'ckpts/occvae_latest.pth'

# 原始 nuScenes 数据和 pkl
DATA_PATH = './data/nuscenes'
TRAIN_IMAGESET = './data/nuscenes_infos_train_temporal_v3_scene.pkl'
VAL_IMAGESET = './data/nuscenes_infos_val_temporal_v3_scene.pkl'

# 训练相关
CUDA_VISIBLE_DEVICES = '3,5 '
SEED = '42'
EMA = 'True'
TB_DIR = './work_dir/dome_flow_no_resample/tb_log'

# 如果从已有 DOME 权重微调，填这个；从头训练就保持空字符串。
LOAD_FROM = ''

# 如果从中断 checkpoint 继续训练，填这个；不继续就保持空字符串。
RESUME_FROM = ''

# 评估/可视化相关
DOME_CKPT = './work_dir/dome_flow_no_resample/latest.pth'
SCENE_IDX = '6 7'
NUM_SAMPLING_STEPS = '20'
VIS_DIR_NAME = 'vis_flow_no_resample'

print('CFG:', CFG)
print('WORK_DIR:', WORK_DIR)
print('VAE_CKPT:', VAE_CKPT)
print('DATA_PATH:', DATA_PATH)

CFG: ./config/train_dome.py
WORK_DIR: ./work_dir/dome_flow_no_resample
VAE_CKPT: ckpts/occvae_latest.pth
DATA_PATH: ./data/nuscenes


## 2. 定义运行命令的小工具

`run_cmd()` 会打印命令和环境变量，然后执行。长时间任务，比如训练，会直接把日志输出在 notebook 里。

如果你只是想先看命令，把 `dry_run=True`。

In [3]:
def run_cmd(cmd, env=None, dry_run=False, allow_error=False):
    env_all = os.environ.copy()
    if env:
        env_all.update({k: str(v) for k, v in env.items() if v is not None})
    print('\n命令:')
    print(' '.join(shlex.quote(str(x)) for x in cmd))
    if env:
        print('\n环境变量:')
        for k, v in env.items():
            if v not in (None, ''):
                print(f'{k}={v}')
    if dry_run:
        print('\n这是 dry run，没有真正执行。')
        return None
    result = subprocess.run(cmd, env=env_all, check=not allow_error)
    if allow_error and result.returncode != 0:
        print(f'\n命令返回码: {result.returncode}')
    return result


def latest_checkpoint(work_dir, preferred):
    preferred = Path(preferred)
    if preferred.exists():
        return str(preferred)
    candidates = sorted(Path(work_dir).glob('epoch_*.pth'), key=lambda p: p.stat().st_mtime)
    if candidates:
        return str(candidates[-1])
    return str(preferred)

## 3. 检查训练前置条件

这个检查只看原始数据路径、train/val pkl、OccVAE checkpoint 和训练入口。它不会检查 `data/resampled_occ`。

In [4]:
required_paths = {
    '训练入口': 'tools/train_diffusion.py',
    '配置文件': CFG,
    'OccVAE checkpoint': VAE_CKPT,
    'nuScenes 数据目录': DATA_PATH,
    'train pkl': TRAIN_IMAGESET,
    'val pkl': VAL_IMAGESET,
}

missing = []
for name, path in required_paths.items():
    exists = Path(path).exists()
    print(f'[{"OK" if exists else "缺失"}] {name}: {path}')
    if not exists:
        missing.append((name, path))

if missing:
    print('\n缺少上面标记为 [缺失] 的文件或目录。路径准备好后再训练。')
else:
    print('\n前置路径检查通过。')

[OK] 训练入口: tools/train_diffusion.py
[OK] 配置文件: ./config/train_dome.py
[OK] OccVAE checkpoint: ckpts/occvae_latest.pth
[OK] nuScenes 数据目录: ./data/nuscenes
[OK] train pkl: ./data/nuscenes_infos_train_temporal_v3_scene.pkl
[OK] val pkl: ./data/nuscenes_infos_val_temporal_v3_scene.pkl

前置路径检查通过。


## 4. 确认配置没有使用 resample

这一步会读取 `config/train_dome.py`，确认训练集类型是 `nuScenesSceneDatasetLidar`，并且数据路径不是 `data/resampled_occ`。

In [25]:
cfg_dict = runpy.run_path(CFG)
train_cfg = cfg_dict['train_dataset_config']
sample_cfg = cfg_dict['sample']

print('train_dataset_config.type:', train_cfg.get('type'))
print('train_dataset_config.data_path:', train_cfg.get('data_path'))
print('train_dataset_config.imageset:', train_cfg.get('imageset'))
print('sample.sample_method:', sample_cfg.get('sample_method'))
print('sample.num_sampling_steps:', sample_cfg.get('num_sampling_steps'))

assert train_cfg.get('type') == 'nuScenesSceneDatasetLidar', '当前配置仍然在使用 resample dataset，请检查 CFG。'
assert 'resampled_occ' not in str(train_cfg.get('data_path')), '当前配置仍然指向 data/resampled_occ，请检查 CFG。'
assert sample_cfg.get('sample_method') == 'flow', '当前配置没有启用 flow matching。'

print('\n确认完成：这个训练版本不使用 resample。')

train_dataset_config.type: nuScenesSceneDatasetLidar
train_dataset_config.data_path: data/nuscenes/
train_dataset_config.imageset: data/nuscenes_infos_train_temporal_v3_scene.pkl
sample.sample_method: flow
sample.num_sampling_steps: 20

确认完成：这个训练版本不使用 resample。


## 5. 启动训练

这一步直接运行 `tools/train_diffusion.py`：

```bash
python tools/train_diffusion.py \
  --py-config ./config/train_dome.py \
  --work-dir ./work_dir/dome_flow_no_resample \
  --vae_load_from ckpts/occvae_latest.pth
```

关键点：OccVAE 会从 `VAE_CKPT` 加载，并在训练脚本中冻结；DOME / world model 才会更新参数。

In [ ]:
RUN_TRAIN = True  # 真正开始训练时改成 True

train_cmd = [
    'python', 'tools/train_diffusion.py',
    '--py-config', CFG,
    '--work-dir', WORK_DIR,
    '--vae_load_from', VAE_CKPT,
    '--seed', SEED,
    '--ema', EMA,
]

if LOAD_FROM:
    train_cmd += ['--load_from', LOAD_FROM]
if RESUME_FROM:
    train_cmd += ['--resume-from', RESUME_FROM]
if TB_DIR:
    train_cmd += ['--tb-dir', TB_DIR]

train_env = {
    'CUDA_VISIBLE_DEVICES': CUDA_VISIBLE_DEVICES,
}

if RUN_TRAIN:
    run_cmd(train_cmd, env=train_env)
else:
    run_cmd(train_cmd, env=train_env, dry_run=True)
    print('\n确认路径没问题后，把 RUN_TRAIN 改成 True 再运行这个 cell。')


命令:
python tools/train_diffusion.py --py-config ./config/train_dome.py --work-dir ./work_dir/dome_flow_no_resample --vae_load_from ckpts/occvae_latest.pth --seed 42 --ema True --tb-dir ./work_dir/dome_flow_no_resample/tb_log

环境变量:
CUDA_VISIBLE_DEVICES=3,5 
Namespace(ema=True, gpus=2, iter_resume=False, load_from=None, py_config='./config/train_dome.py', resume_from='', seed=42, tb_dir='./work_dir/dome_flow_no_resample/tb_log', vae_load_from='ckpts/occvae_latest.pth', work_dir='./work_dir/dome_flow_no_resample')
tcp://127.0.0.1:25098
tcp://127.0.0.1:25098
04/14 18:05:22 - mmengine - INFO - Config:
_dim_ = 16
base_channel = 64
cond_frames_choices = [
    [],
    [
        0,
    ],
    [
        0,
        1,
    ],
    [
        0,
        1,
        2,
    ],
    [
        0,
        1,
        2,
        3,
    ],
]
data_path = 'data/nuscenes/'
ema = True
end_frame = 10
eval_every_epochs = 10
eval_length = 6
expansion = 8
find_unused_parameters = True
gpu_ids = range(0, 2)
grad_max_

[W reducer.cpp:1300] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later iterations to have unused parameters. (function operator())
[W reducer.cpp:1300] Warning: find_unused_parameters=True was specified in DDP constructor, but did not find any unused parameters in the forward pass. This flag results in an extra traversal of the autograd graph every iteration,  which can adversely affect performance. If your model indeed never has any unused parameters in the forward pass, consider turning this flag off. Note that this warning may be a false positive if your model has flow control causing later 

04/14 18:06:54 - mmengine - INFO - [TRAIN] Epoch 0 Iter     0/219: Loss: 0.806 (0.806), grad_norm: 0.478, lr: 0.0001000, time: 45.772 (38.000)
04/14 18:06:54 - mmengine - INFO - mse: 0.80561, loss: 0.80561
04/14 18:06:55 - mmengine - INFO - [TRAIN] Epoch 0 Iter     1/219: Loss: 0.836 (0.836), grad_norm: 0.454, lr: 0.0001000, time: 1.794 (0.023)
04/14 18:06:55 - mmengine - INFO - mse: 0.83555, loss: 0.83555
04/14 18:06:57 - mmengine - INFO - [TRAIN] Epoch 0 Iter     2/219: Loss: 0.784 (0.784), grad_norm: 0.379, lr: 0.0001000, time: 1.858 (0.018)
04/14 18:06:57 - mmengine - INFO - mse: 0.78412, loss: 0.78412
04/14 18:06:59 - mmengine - INFO - [TRAIN] Epoch 0 Iter     3/219: Loss: 0.755 (0.755), grad_norm: 0.352, lr: 0.0001000, time: 1.911 (0.029)
04/14 18:06:59 - mmengine - INFO - mse: 0.75480, loss: 0.75480
04/14 18:07:01 - mmengine - INFO - [TRAIN] Epoch 0 Iter     4/219: Loss: 0.682 (0.682), grad_norm: 0.261, lr: 0.0001000, time: 1.837 (0.125)
04/14 18:07:01 - mmengine - INFO - mse: 0

## 6. TensorBoard 看训练指标

训练脚本默认会把 TensorBoard 日志写到 `WORK_DIR/tb_log`。这个 notebook 里设置的是：

```text
./work_dir/dome_flow_no_resample/tb_log
```

主要看这些指标：

- `train/loss`
- `train/mse`
- `train/lr`
- `train/grad_norm`
- `val/iou`
- `val/miou`

在服务器上通常用 `--host 0.0.0.0`，然后通过端口转发或服务器面板打开。

In [ ]:
print('TensorBoard 日志目录:', TB_DIR)
print('启动命令:')
print(f'tensorboard --logdir {shlex.quote(TB_DIR)} --host 0.0.0.0 --port 6006')

## 7. 评估 checkpoint

训练完成或保存了某个 epoch 后，运行评估脚本。

如果 `DOME_CKPT` 不存在，下面的 helper 会从 `WORK_DIR` 里找最新的 `epoch_*.pth`。

In [ ]:
RUN_EVAL = False  # 要评估时改成 True

eval_ckpt = latest_checkpoint(WORK_DIR, DOME_CKPT)
eval_cmd = [
    'python', 'tools/eval_metric.py',
    '--py-config', CFG,
    '--work-dir', WORK_DIR,
    '--resume-from', eval_ckpt,
    '--vae-resume-from', VAE_CKPT,
    '--seed', SEED,
]

eval_env = {
    'CUDA_VISIBLE_DEVICES': CUDA_VISIBLE_DEVICES,
}

if RUN_EVAL:
    run_cmd(eval_cmd, env=eval_env)
else:
    run_cmd(eval_cmd, env=eval_env, dry_run=True)
    print('\n需要评估时，把 RUN_EVAL 改成 True。')

## 8. 可视化预测结果

这一步会把预测 occupancy 画出来，并生成视频。

默认看 scene 6 和 scene 7。你可以改 `SCENE_IDX`，比如：

```python
SCENE_IDX = '6 7 16 18'
```

`NUM_SAMPLING_STEPS` 是 flow matching 采样步数，越大通常越慢。

In [ ]:
RUN_VIS = False  # 要可视化时改成 True

vis_ckpt = latest_checkpoint(WORK_DIR, DOME_CKPT)
scene_idx_args = SCENE_IDX.split()
vis_cmd = [
    'python', 'tools/visualize_demo.py',
    '--py-config', CFG,
    '--work-dir', WORK_DIR,
    '--resume-from', vis_ckpt,
    '--vae-resume-from', VAE_CKPT,
    '--dir-name', VIS_DIR_NAME,
    '--num_sampling_steps', NUM_SAMPLING_STEPS,
    '--seed', SEED,
    '--scene-idx', *scene_idx_args,
]

vis_env = {
    'CUDA_VISIBLE_DEVICES': CUDA_VISIBLE_DEVICES,
}

if RUN_VIS:
    run_cmd(vis_cmd, env=vis_env)
else:
    run_cmd(vis_cmd, env=vis_env, dry_run=True)
    print('\n需要可视化时，把 RUN_VIS 改成 True。')

## 9. 训练过程中到底发生了什么

一次训练迭代可以理解成：

```text
1. 从原始 data/nuscenes 和 train pkl 里取 11 帧 occupancy
2. OccVAE encoder 把 occupancy 压缩成 latent
3. 随机造一份噪声 x0
4. 真实 latent 当作 x1
5. flow matching 在 x0 和 x1 中间采样 x_t
6. DOME 根据 x_t、时间 t、前 4 帧条件和 pose 预测速度方向
7. 用 MSE 比较预测速度和真实速度
8. 只更新 DOME 参数
9. OccVAE 不更新
```

这个版本和 resample 版本的核心区别只有数据来源：

```text
无 resample：data/nuscenes + nuScenesSceneDatasetLidar
有 resample：data/resampled_occ + nuScenesSceneDatasetLidarResample
```

## 10. 最常用命令汇总

如果不想用 notebook，也可以直接在终端跑：

```bash
cd /mnt/data2/whz/dome-cfm
conda activate torchcfm

CUDA_VISIBLE_DEVICES=0 python tools/train_diffusion.py \
  --py-config ./config/train_dome.py \
  --work-dir ./work_dir/dome_flow_no_resample \
  --vae_load_from ckpts/occvae_latest.pth \
  --seed 42 \
  --ema True

tensorboard --logdir ./work_dir/dome_flow_no_resample/tb_log --host 0.0.0.0 --port 6006

CUDA_VISIBLE_DEVICES=0 python tools/eval_metric.py \
  --py-config ./config/train_dome.py \
  --work-dir ./work_dir/dome_flow_no_resample \
  --resume-from ./work_dir/dome_flow_no_resample/latest.pth \
  --vae-resume-from ckpts/occvae_latest.pth

CUDA_VISIBLE_DEVICES=0 python tools/visualize_demo.py \
  --py-config ./config/train_dome.py \
  --work-dir ./work_dir/dome_flow_no_resample \
  --resume-from ./work_dir/dome_flow_no_resample/latest.pth \
  --vae-resume-from ckpts/occvae_latest.pth \
  --dir-name vis_flow_no_resample \
  --scene-idx 6 7
```